# 🚑 Ambulance Dispatch — OpenEnv RL Training Notebook

**OpenEnv Hackathon** | Scaler × Meta × HuggingFace × PyTorch

This notebook demonstrates GRPO training on the Ambulance-OpenENV environment using HuggingFace TRL + Unsloth.

> **For the full GRPO training notebook** with Unsloth 4-bit quantisation, training curves, and before/after evaluation,
> open **`notebooks/grpo_colab.ipynb`** in Colab.

This notebook provides a **quick-start DQN demo** to verify the environment works end-to-end.

---


In [ ]:
# Cell 1 — Install dependencies and clone repo
import subprocess, sys, os

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip("matplotlib", "numpy", "networkx", "pydantic>=2.0", "fastapi", "uvicorn")

# Clone the repo if running in Colab
if not os.path.exists("Ambulance-OpenENV"):
    subprocess.check_call([
        "git", "clone",
        "https://github.com/CSNEHA20/Meta_PyTorch_OpenEnv_Hackathon.git",
        "Ambulance-OpenENV"
    ])

if os.path.basename(os.getcwd()) != "Ambulance-OpenENV":
    os.chdir("Ambulance-OpenENV")

# Add repo root to path
import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Working directory: {os.getcwd()}")


In [ ]:
# Cell 2 — Imports
import json
import random
import numpy as np
import matplotlib.pyplot as plt

from env.environment import AmbulanceEnvironment
from env.models import ActionModel, AmbulanceState, Severity

print("Imports OK")


In [ ]:
# Cell 3 — Initialise environment
ENV_CONFIG = {
    "n_ambulances": 3,
    "n_hospitals": 2,
    "max_steps": 30,
    "seed": 42,
}

env = AmbulanceEnvironment(ENV_CONFIG)
obs = env.reset()

print(f"Env ready:  {len(obs.ambulances)} ambulances, {len(obs.hospitals)} hospitals")
print(f"Emergencies at reset: {len(obs.emergencies)}")
print(f"Step: {obs.step}, Done: {obs.done}")


In [ ]:
# Cell 4 — Greedy dispatch agent (rule-based baseline)
def greedy_action(obs):
    """Pick the highest-severity unassigned emergency and nearest idle ambulance."""
    idle_ambs = [a for a in obs.ambulances if a.state == AmbulanceState.IDLE]
    open_emgs = sorted(
        [e for e in obs.emergencies if not e.assigned],
        key=lambda e: {"CRITICAL": 0, "HIGH": 1, "NORMAL": 2}.get(e.severity.value, 3)
    )
    open_hosps = sorted(obs.hospitals, key=lambda h: h.current_patients)

    if idle_ambs and open_emgs and open_hosps:
        return ActionModel(
            ambulance_id=idle_ambs[0].id,
            emergency_id=open_emgs[0].id,
            hospital_id=open_hosps[0].id,
        )
    return ActionModel(ambulance_id=None, emergency_id="", is_noop=True)


def run_episode(agent_fn, config=None, seed=None):
    """Run one full episode; return total reward and per-step rewards."""
    cfg = config or ENV_CONFIG
    e = AmbulanceEnvironment(cfg)
    o = e.reset(seed=seed)
    rewards = []
    while not o.done:
        action = agent_fn(o)
        o = e.step(action)
        rewards.append(o.reward)
    return sum(rewards), rewards


# Sanity-check one greedy episode
total, per_step = run_episode(greedy_action)
print(f"Greedy episode reward: {total:.2f}  ({len(per_step)} steps)")


In [ ]:
# Cell 5 — Random baseline for comparison
def random_action(obs):
    """Dispatch a random idle ambulance to a random emergency."""
    idle_ambs = [a for a in obs.ambulances if a.state == AmbulanceState.IDLE]
    open_emgs = [e for e in obs.emergencies if not e.assigned]
    hosps = obs.hospitals
    if idle_ambs and open_emgs and hosps:
        return ActionModel(
            ambulance_id=random.choice(idle_ambs).id,
            emergency_id=random.choice(open_emgs).id,
            hospital_id=random.choice(hosps).id,
        )
    return ActionModel(ambulance_id=None, emergency_id="", is_noop=True)


N_EVAL = 20
random.seed(0)

random_rewards  = [run_episode(random_action,  seed=s)[0] for s in range(N_EVAL)]
greedy_rewards  = [run_episode(greedy_action,  seed=s)[0] for s in range(N_EVAL)]

print(f"Random agent  — avg reward: {np.mean(random_rewards):.2f}  ± {np.std(random_rewards):.2f}")
print(f"Greedy agent  — avg reward: {np.mean(greedy_rewards):.2f}  ± {np.std(greedy_rewards):.2f}")
print(f"Improvement   : {np.mean(greedy_rewards) - np.mean(random_rewards):+.2f}")


In [ ]:
# Cell 6 — Simple Q-learning agent (tabular, for demo purposes)
from collections import defaultdict

class TabularQAgent:
    """Simple tabular Q-agent over a discretised state (n_idle, n_emg, step_bin)."""

    def __init__(self, lr=0.1, gamma=0.95, eps=0.3):
        self.Q = defaultdict(float)
        self.lr = lr
        self.gamma = gamma
        self.eps = eps

    def _state_key(self, obs):
        n_idle = sum(1 for a in obs.ambulances if a.state == AmbulanceState.IDLE)
        n_emg  = sum(1 for e in obs.emergencies if not e.assigned)
        step_bin = min(obs.step // 5, 5)
        return (n_idle, n_emg, step_bin)

    def _action_key(self, obs):
        """Two possible meta-actions: dispatch or noop."""
        idle_ambs = [a for a in obs.ambulances if a.state == AmbulanceState.IDLE]
        open_emgs = [e for e in obs.emergencies if not e.assigned]
        return "dispatch" if (idle_ambs and open_emgs) else "noop"

    def act(self, obs):
        ak = self._action_key(obs)
        if ak == "noop" or random.random() < self.eps:
            return greedy_action(obs)   # fallback to greedy
        # Execute greedy dispatch
        return greedy_action(obs)

    def update(self, sk, ak, reward, sk_next):
        old = self.Q[(sk, ak)]
        best_next = max(self.Q[(sk_next, a)] for a in ("dispatch", "noop"))
        self.Q[(sk, ak)] = old + self.lr * (reward + self.gamma * best_next - old)


def train_q_agent(n_episodes=200):
    agent = TabularQAgent()
    episode_rewards = []
    for ep in range(n_episodes):
        e = AmbulanceEnvironment(ENV_CONFIG)
        obs = e.reset(seed=ep)
        total_r = 0.0
        agent.eps = max(0.05, 0.3 - ep * 0.001)  # decay exploration
        while not obs.done:
            sk = agent._state_key(obs)
            action = agent.act(obs)
            ak = agent._action_key(obs)
            obs_next = e.step(action)
            r = obs_next.reward
            sk_next = agent._state_key(obs_next)
            agent.update(sk, ak, r, sk_next)
            total_r += r
            obs = obs_next
        episode_rewards.append(total_r)
    return agent, episode_rewards


print("Training tabular Q-agent for 200 episodes...")
q_agent, train_rewards = train_q_agent(200)
print(f"Final 20-ep avg reward: {np.mean(train_rewards[-20:]):.2f}")


In [ ]:
# Cell 7 — Evaluate trained Q-agent vs baselines
def q_agent_fn(agent):
    def fn(obs):
        return agent.act(obs)
    return fn

q_eval_rewards = [run_episode(q_agent_fn(q_agent), seed=s)[0] for s in range(N_EVAL)]

print(f"Random agent  — avg reward: {np.mean(random_rewards):.2f}")
print(f"Greedy agent  — avg reward: {np.mean(greedy_rewards):.2f}")
print(f"Q-agent       — avg reward: {np.mean(q_eval_rewards):.2f}")


In [ ]:
# Cell 8 — Plot training curve and before/after comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Ambulance Dispatch — Training Demo", fontsize=14, fontweight="bold")

# Left: Q-agent training reward curve with 20-ep moving average
window = 20
ma = np.convolve(train_rewards, np.ones(window) / window, mode="valid")
ep_range = np.arange(len(train_rewards))
ma_range = np.arange(window - 1, len(train_rewards))

axes[0].plot(ep_range, train_rewards, alpha=0.3, color="#6366f1", linewidth=1, label="Per-episode")
axes[0].plot(ma_range, ma, color="#6366f1", linewidth=2.5, label=f"{window}-ep moving avg")
axes[0].set_title("Q-Agent Training Reward Curve")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total Reward")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: bar chart — Random vs Greedy vs Q-Agent
agents = ["Random", "Greedy", "Q-Agent"]
avgs   = [np.mean(random_rewards), np.mean(greedy_rewards), np.mean(q_eval_rewards)]
colors = ["#ef4444", "#f59e0b", "#10b981"]

bars = axes[1].bar(agents, avgs, color=colors, alpha=0.85, width=0.5)
for bar, val in zip(bars, avgs):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f"{val:.1f}", ha="center", va="bottom", fontweight="bold")
axes[1].set_title("Agent Comparison (avg over 20 episodes)")
axes[1].set_ylabel("Avg Episode Reward")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("reward_curve.png", dpi=150)
plt.show()
print("Saved reward_curve.png")


In [ ]:
# Cell 9 — Run the official graders for Easy / Medium / Hard
import importlib, sys

def run_grader(module_name):
    mod = importlib.import_module(module_name)
    return mod.grade()

scores = {}
for name, mod in [("Easy", "tasks.easy"), ("Medium", "tasks.medium"), ("Hard", "tasks.hard")]:
    try:
        score = run_grader(mod)
        scores[name] = score
        print(f"{name:8s} score: {score:.4f}")
    except Exception as ex:
        print(f"{name:8s} grader error: {ex}")

print("\nReproducible scores (seed=42):")
for k, v in scores.items():
    print(f"  {k}: {v:.3f}")


In [ ]:
# Cell 10 — RFC 004 Rubric breakdown (one greedy episode)
env2 = AmbulanceEnvironment(ENV_CONFIG)
obs2 = env2.reset(seed=42)
rubric_totals = {}

while not obs2.done:
    action = greedy_action(obs2)
    obs2 = env2.step(action)
    if obs2.rubric:
        for k, v in obs2.rubric.model_dump().items():
            rubric_totals[k] = rubric_totals.get(k, 0.0) + v

if rubric_totals:
    components = list(rubric_totals.keys())
    values = [rubric_totals[c] for c in components]
    colors = ["#10b981" if v >= 0 else "#ef4444" for v in values]

    plt.figure(figsize=(10, 4))
    plt.bar(components, values, color=colors, alpha=0.85)
    plt.axhline(0, color="black", linewidth=0.8)
    plt.title("RFC 004 Rubric — Cumulative Component Breakdown (Greedy Agent, seed=42)")
    plt.ylabel("Cumulative Reward")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig("rubric_breakdown.png", dpi=150)
    plt.show()
    print("Saved rubric_breakdown.png")
else:
    print("No rubric data — rubric may be None for this config")


## Next Steps

This notebook provides a working end-to-end demo using a simple Q-agent.

For the **full GRPO training pipeline** with Unsloth 4-bit quantisation, LLM prompt engineering, and before/after evaluation plots, open:

👉 **[`notebooks/grpo_colab.ipynb`](notebooks/grpo_colab.ipynb)**

```
Repository : https://github.com/CSNEHA20/Meta_PyTorch_OpenEnv_Hackathon
HF Space   : https://huggingface.co/spaces/vishallakshmikanthan/Ambulance-OpenENV
```

Run the server locally:
```bash
uvicorn server.app:app --host 0.0.0.0 --port 7860
```
